# Can we predict 12-month loan default?
Target: `defaulted` (1 = defaulted within 12 months of funding). Features are all measured **at application**, so there's no obvious leakage; `application_id` is an identifier and is excluded.

Plan: data checks → EDA → baseline vs. logistic regression vs. gradient boosting (cross-validated on a train set) → one-time evaluation on a held-out test set → calibration, importance, and a practical lift/threshold view.

In [ ]:
import os, warnings
os.environ.setdefault('MPLCONFIGDIR', '/tmp/mplcfg')
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve, precision_recall_curve
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
SEED=42
df=pd.read_csv('data.csv'); cols=pd.read_csv('columns.csv')
df.shape

## 1. Data checks

In [ ]:
print(df.isna().sum().sum(), 'missing |', df.duplicated().sum(), 'duplicate rows |', df.application_id.duplicated().sum(), 'duplicate ids')
print(df.defaulted.value_counts().rename('n').to_frame().assign(share=lambda x: x.n/len(df)))
df.describe().T

In [ ]:
# plausibility: tenure longer than adult life, extreme values
print('employment_years > age-16:', (df.employment_years > df.age-16).sum())
print('zero employment_years:', (df.employment_years==0).sum())
print('debt_ratio>0.8:', (df.debt_ratio>0.8).sum(), '| income_k min:', df.income_k.min())

## 2. Exploration

In [ ]:
num=['income_k','debt_ratio','credit_lines','age','employment_years','loan_amount_k']
fig,ax=plt.subplots(2,3,figsize=(14,7))
for a,c in zip(ax.ravel(),num):
    q=pd.qcut(df[c],10,duplicates='drop')
    r=df.groupby(q,observed=True).defaulted.agg(['mean','size'])
    a.plot(range(len(r)),r['mean'],marker='o'); a.axhline(df.defaulted.mean(),ls='--',c='grey')
    a.set_title(f'Default rate by {c} decile'); a.set_xlabel('decile (low→high)'); a.set_ylabel('default rate')
plt.tight_layout(); plt.show()
print(df.groupby('region').defaulted.agg(['mean','size']))

Default rate rises with `debt_ratio` and `loan_amount_k` and falls with `income_k` and `employment_years`. `age`, `credit_lines` and `region` look flat on their own (we'll test whether that holds up in the models).

## 3. Split and models
Stratified 80/20 split. The test set is touched once, at the end. Model choice uses 5-fold CV on the training set. No time column is available, so a random split is the only option (see caveats).

In [ ]:
X=df.drop(columns=['defaulted','application_id']); y=df.defaulted
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.2,stratify=y,random_state=SEED)
cat=['region']; num=[c for c in X if c not in cat]
pre=ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(drop='first'),cat)])
models={
 'baseline (prevalence)':DummyClassifier(strategy='prior'),
 'logistic regression':make_pipeline(pre,LogisticRegression(max_iter=1000)),
 'gradient boosting':HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,min_samples_leaf=50,l2_regularization=1.0,categorical_features='from_dtype',random_state=SEED),
}
Xtr_g=Xtr.assign(region=Xtr.region.astype('category')); Xte_g=Xte.assign(region=pd.Categorical(Xte.region,categories=Xtr_g.region.cat.categories))
skf=StratifiedKFold(5,shuffle=True,random_state=SEED)
oof={}; rows=[]
for n,m in models.items():
    Xs=Xtr_g if 'boosting' in n else Xtr
    p=cross_val_predict(m,Xs,ytr,cv=skf,method='predict_proba')[:,1]; oof[n]=p
    rows.append((n,roc_auc_score(ytr,p),average_precision_score(ytr,p),brier_score_loss(ytr,p)))
cv=pd.DataFrame(rows,columns=['model','ROC AUC','PR AUC','Brier']).set_index('model').round(4)
print(f'Prevalence (= PR AUC of a random model): {ytr.mean():.3f}'); cv

## 4. Held-out test performance

In [ ]:
fitted={}
for n,m in models.items():
    fitted[n]=m.fit(Xtr_g if 'boosting' in n else Xtr, ytr)
pt={n:fitted[n].predict_proba(Xte_g if 'boosting' in n else Xte)[:,1] for n in models}
rng=np.random.default_rng(SEED)
def boot_auc(p,B=1000):
    idx=[rng.integers(0,len(yte),len(yte)) for _ in range(B)]
    a=[roc_auc_score(yte.values[i],p[i]) for i in idx]; return np.percentile(a,[2.5,97.5])
res=[]
for n,p in pt.items():
    lo,hi=boot_auc(p); res.append((n,roc_auc_score(yte,p),f'[{lo:.3f}, {hi:.3f}]',average_precision_score(yte,p),brier_score_loss(yte,p)))
pd.DataFrame(res,columns=['model','ROC AUC','95% CI (bootstrap)','PR AUC','Brier']).set_index('model').round(4)

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(16,4.5))
for n,p in pt.items():
    f,t,_=roc_curve(yte,p); ax[0].plot(f,t,label=n)
    pr,rc,_=precision_recall_curve(yte,p); ax[1].plot(rc,pr,label=n)
ax[0].plot([0,1],[0,1],'k:'); ax[0].set(title='ROC',xlabel='FPR',ylabel='TPR'); ax[0].legend()
ax[1].set(title='Precision-recall',xlabel='recall',ylabel='precision')
for n in ['logistic regression','gradient boosting']:
    fr,mp=calibration_curve(yte,pt[n],n_bins=8,strategy='quantile'); ax[2].plot(mp,fr,marker='o',label=n)
ax[2].plot([0,.5],[0,.5],'k:'); ax[2].set(title='Calibration',xlabel='predicted',ylabel='observed'); ax[2].legend()
plt.tight_layout(); plt.show()

## 5. What drives the prediction?

In [ ]:
lr=fitted['logistic regression']
coef=pd.Series(lr[-1].coef_[0],index=lr[0].get_feature_names_out()).sort_values()
print('Logistic coefficients (per 1 SD for numeric features):'); print(coef.round(3).to_string())
pi=permutation_importance(fitted['gradient boosting'],Xte_g,yte,scoring='roc_auc',n_repeats=20,random_state=SEED)
pd.DataFrame({'AUC drop':pi.importances_mean,'sd':pi.importances_std},index=Xte_g.columns).sort_values('AUC drop',ascending=False).round(4)

## 6. Practical value: ranking and thresholds

In [ ]:
best='logistic regression' if roc_auc_score(yte,pt['logistic regression'])>=roc_auc_score(yte,pt['gradient boosting']) else 'gradient boosting'
p=pt[best]; d=pd.DataFrame({'p':p,'y':yte.values})
d['decile']=pd.qcut(d.p.rank(method='first'),10,labels=range(1,11))
lift=d.groupby('decile',observed=True).agg(n=('y','size'),default_rate=('y','mean'),mean_pred=('p','mean')).round(3)
lift['lift']=(lift.default_rate/yte.mean()).round(2)
print('Model shown:',best); print(lift.sort_index(ascending=False))
for k in (0.1,0.2,0.3):
    top=d.nlargest(int(k*len(d)),'p'); print(f'Top {k:.0%} riskiest captures {top.y.sum()/d.y.sum():.0%} of defaults (precision {top.y.mean():.0%})')

## Conclusion

In [ ]:
cvr=cv.loc[['logistic regression','gradient boosting'],'ROC AUC']
print(f"CV ROC AUC: LR {cvr.iloc[0]:.3f}, GB {cvr.iloc[1]:.3f}; test ROC AUC ({best}): {roc_auc_score(yte,p):.3f}")

**Findings** (figures from the run that produced this notebook; re-running with the same seed reproduces them):
- **Yes, moderately.** Test ROC AUC ≈ 0.775 (bootstrap 95% CI ≈ 0.74–0.81) vs. 0.50 for the baseline; CV agrees (0.775). It ranks risk well but is not a sharp classifier: with ~10.5% prevalence, the riskiest 10% of applicants default ~36% of the time (≈3.5× lift) and contain ~35% of all defaults; the riskiest 30% contain ~63%.
- **Simple beats complex.** Regularised logistic regression (AUC 0.775) is at least as good as gradient boosting (0.761), so the signal is mostly additive and monotone. Prefer the interpretable model.
- **Drivers:** higher `debt_ratio` (strongest) and `loan_amount_k` raise risk; higher `income_k` and `employment_years` lower it. `age`, `credit_lines` and `region` contribute essentially nothing (permutation importance ≈ 0).
- **Calibration** is reasonable for the logistic model (predicted ≈ observed by decile), so scores can be used as probabilities, though the lower deciles are noisy with only 160 loans each.

**Data quality flag**
- 643 rows have `employment_years` greater than `age − 16`, and 720 have exactly 0. Possibly a data-entry or default-value artifact (0 may mean "unknown"). Worth checking with the data owner; I left the values unchanged.

**Caveats**
- Random split: there's no application date, so I couldn't test on a later period. Drift could lower performance; validate out-of-time before relying on it.
- Outcomes exist only for funded loans, so the model describes funded applicants (selection bias) and may not transfer to those who would have been declined.
- The test set has ~170 defaults, so metrics carry real uncertainty (see the CI).
- The right threshold depends on the cost of a missed default vs. a wrongly declined loan, which isn't specified here.
- Using `region`/`age` in actual credit decisions may be restricted by fair-lending rules; they add no predictive value here, so they could be dropped anyway.